# Testes interativos da camada de persistência — Oficina Mecânica

Este notebook avalia o funcionamento do mapeamento ORM alinhado aos diagramas de classes e E-R do projeto.

**Relações validadas**
- **1:N** — `Cliente` → `Veiculo`
- **1:1** — `OrdemServico` ↔ `NotaFiscal`
- **N:M** — `OrdemServico` ↔ `Mecanico` (tabela associativa `ordem_mecanico`)

A camada oficial em Java/JPA está em `src/main/java`. Aqui usamos **SQLAlchemy** sobre o mesmo esquema E-R para testes interativos no Jupyter.


## 1. Setup e conexão (SQLite em memória)

In [ ]:
from datetime import date
from sqlalchemy import (
    create_engine, Column, Integer, BigInteger, String, Float, Date, ForeignKey, Table,
    select, func
)
from sqlalchemy.orm import declarative_base, relationship, sessionmaker, Session

engine = create_engine("sqlite:///:memory:", echo=False)
SessionLocal = sessionmaker(bind=engine, expire_on_commit=False)
Base = declarative_base()
print("Engine OK:", engine.url)

## 2. Modelagem ORM (mesmo E-R dos diagramas)

In [ ]:
ordem_mecanico = Table(
    "ordem_mecanico",
    Base.metadata,
    Column("ordem_id", ForeignKey("ordem_servico.id"), primary_key=True),
    Column("mecanico_id", ForeignKey("mecanico.id"), primary_key=True),
)


class Cliente(Base):
    __tablename__ = "cliente"
    id = Column(Integer, primary_key=True, autoincrement=True)
    nome = Column(String(100), nullable=False)
    cpf_cnpj = Column(String(20), nullable=False, unique=True)
    endereco = Column(String(255))
    cep = Column(String(10))
    numero = Column(String(10))
    complemento = Column(String(100))
    veiculos = relationship("Veiculo", back_populates="cliente", cascade="all, delete-orphan")

    def __repr__(self):
        return f"Cliente(id={self.id}, nome={self.nome!r})"


class Veiculo(Base):
    __tablename__ = "veiculo"
    id = Column(Integer, primary_key=True, autoincrement=True)
    placa = Column(String(10), nullable=False, unique=True)
    modelo = Column(String(50))
    ano_modelo = Column(Integer)
    cliente_id = Column(Integer, ForeignKey("cliente.id"), nullable=False)
    cliente = relationship("Cliente", back_populates="veiculos")
    ordens = relationship("OrdemServico", back_populates="veiculo", cascade="all, delete-orphan")

    def __repr__(self):
        return f"Veiculo(id={self.id}, placa={self.placa!r})"


class Mecanico(Base):
    __tablename__ = "mecanico"
    id = Column(Integer, primary_key=True, autoincrement=True)
    nome = Column(String(100), nullable=False)
    ordens = relationship("OrdemServico", secondary=ordem_mecanico, back_populates="mecanicos")

    def __repr__(self):
        return f"Mecanico(id={self.id}, nome={self.nome!r})"


class Servico(Base):
    __tablename__ = "servico"
    id = Column(Integer, primary_key=True, autoincrement=True)
    descricao = Column(String(150), nullable=False)
    valor = Column(Float, nullable=False)

    def __repr__(self):
        return f"Servico(id={self.id}, descricao={self.descricao!r}, valor={self.valor})"


class Peca(Base):
    __tablename__ = "peca"
    id = Column(Integer, primary_key=True, autoincrement=True)
    nome = Column(String(100), nullable=False)
    preco_unitario = Column(Float, nullable=False)

    def __repr__(self):
        return f"Peca(id={self.id}, nome={self.nome!r})"


class OrdemServico(Base):
    __tablename__ = "ordem_servico"
    id = Column(Integer, primary_key=True, autoincrement=True)
    data_emissao = Column(Date, nullable=False)
    valor_total = Column(Float, default=0.0)
    veiculo_id = Column(Integer, ForeignKey("veiculo.id"), nullable=False)
    veiculo = relationship("Veiculo", back_populates="ordens")
    mecanicos = relationship("Mecanico", secondary=ordem_mecanico, back_populates="ordens")
    itens_servico = relationship("ItemServico", back_populates="ordem", cascade="all, delete-orphan")
    itens_peca = relationship("ItemPeca", back_populates="ordem", cascade="all, delete-orphan")
    nota_fiscal = relationship("NotaFiscal", back_populates="ordem", uselist=False, cascade="all, delete-orphan")

    def recalcular_valor_total(self):
        total_s = sum(i.servico.valor * i.quantidade for i in self.itens_servico)
        total_p = sum(i.peca.preco_unitario * i.quantidade for i in self.itens_peca)
        self.valor_total = total_s + total_p

    def __repr__(self):
        return f"OrdemServico(id={self.id}, valor_total={self.valor_total})"


class NotaFiscal(Base):
    __tablename__ = "nota_fiscal"
    id = Column(Integer, primary_key=True, autoincrement=True)
    numero = Column(String(20), nullable=False, unique=True)
    chave_acesso = Column(String(44))
    data_emissao = Column(Date, nullable=False)
    valor_imposto = Column(Float)
    ordem_id = Column(Integer, ForeignKey("ordem_servico.id"), nullable=False, unique=True)
    ordem = relationship("OrdemServico", back_populates="nota_fiscal")

    def __repr__(self):
        return f"NotaFiscal(id={self.id}, numero={self.numero!r})"


class ItemServico(Base):
    __tablename__ = "item_servico"
    id = Column(Integer, primary_key=True, autoincrement=True)
    quantidade = Column(Integer, nullable=False)
    ordem_id = Column(Integer, ForeignKey("ordem_servico.id"), nullable=False)
    servico_id = Column(Integer, ForeignKey("servico.id"), nullable=False)
    ordem = relationship("OrdemServico", back_populates="itens_servico")
    servico = relationship("Servico")


class ItemPeca(Base):
    __tablename__ = "item_peca"
    id = Column(Integer, primary_key=True, autoincrement=True)
    quantidade = Column(Integer, nullable=False)
    ordem_id = Column(Integer, ForeignKey("ordem_servico.id"), nullable=False)
    peca_id = Column(Integer, ForeignKey("peca.id"), nullable=False)
    ordem = relationship("OrdemServico", back_populates="itens_peca")
    peca = relationship("Peca")


Base.metadata.create_all(engine)
print("Tabelas criadas:", sorted(Base.metadata.tables.keys()))

## 3. Persistência de um cenário completo

In [ ]:
session: Session = SessionLocal()

cliente = Cliente(
    nome="Ana Souza",
    cpf_cnpj="123.456.789-00",
    endereco="Rua das Oficinas",
    cep="74000-000",
    numero="100",
    complemento="Sala 2",
)
veiculo = Veiculo(placa="ABC1D23", modelo="Gol 1.6", ano_modelo=2018, cliente=cliente)

m1 = Mecanico(nome="Carlos Mecanico")
m2 = Mecanico(nome="Diana Especialista")
troca_oleo = Servico(descricao="Troca de oleo", valor=120.0)
alinhamento = Servico(descricao="Alinhamento", valor=90.0)
filtro = Peca(nome="Filtro de oleo", preco_unitario=45.0)
pastilha = Peca(nome="Pastilha de freio", preco_unitario=80.0)

ordem = OrdemServico(data_emissao=date.today(), veiculo=veiculo, mecanicos=[m1, m2])
ordem.itens_servico = [
    ItemServico(servico=troca_oleo, quantidade=1),
    ItemServico(servico=alinhamento, quantidade=1),
]
ordem.itens_peca = [
    ItemPeca(peca=filtro, quantidade=1),
    ItemPeca(peca=pastilha, quantidade=2),
]
ordem.recalcular_valor_total()
ordem.nota_fiscal = NotaFiscal(
    numero="NF-0001",
    chave_acesso="35260900000000000000550010000000011000000001",
    data_emissao=date.today(),
    valor_imposto=round(ordem.valor_total * 0.12, 2),
)

session.add_all([cliente, m1, m2, troca_oleo, alinhamento, filtro, pastilha, ordem])
session.commit()

print("Cliente:", cliente)
print("Veículos (1:N):", cliente.veiculos)
print("Ordem:", ordem)
print("Mecânicos (N:M):", ordem.mecanicos)
print("Nota fiscal (1:1):", ordem.nota_fiscal)
print(f"Valor total: R$ {ordem.valor_total:.2f}")

## 4. Teste 1:N — Cliente possui vários veículos

In [ ]:
v2 = Veiculo(placa="XYZ9K88", modelo="Onix", ano_modelo=2021, cliente=cliente)
session.add(v2)
session.commit()

cliente_db = session.get(Cliente, cliente.id)
assert len(cliente_db.veiculos) == 2
placas = sorted(v.placa for v in cliente_db.veiculos)
print("Veículos do cliente:", placas)
print("OK — relação 1:N Cliente-Veiculo")

## 5. Teste 1:1 — OrdemServico ↔ NotaFiscal

In [ ]:
ordem_db = session.get(OrdemServico, ordem.id)
nota_db = session.get(NotaFiscal, ordem.nota_fiscal.id)

assert ordem_db.nota_fiscal is not None
assert nota_db.ordem_id == ordem_db.id
assert nota_db.ordem.id == ordem_db.id

# unicidade: uma OS não pode ter duas NFs
try:
    session.add(NotaFiscal(
        numero="NF-DUPLICADA",
        chave_acesso="x",
        data_emissao=date.today(),
        valor_imposto=1.0,
        ordem_id=ordem_db.id,
    ))
    session.commit()
    raise AssertionError("Deveria falhar por unique em ordem_id")
except Exception as exc:
    session.rollback()
    print("Bloqueio de segunda NF esperado:", type(exc).__name__)

print("OK — relação 1:1 OrdemServico-NotaFiscal")

## 6. Teste N:M — OrdemServico ↔ Mecanico

In [ ]:
ordem_db = session.get(OrdemServico, ordem.id)
m1_db = session.get(Mecanico, m1.id)
m2_db = session.get(Mecanico, m2.id)

assert {m.nome for m in ordem_db.mecanicos} == {"Carlos Mecanico", "Diana Especialista"}
assert ordem_db in m1_db.ordens
assert ordem_db in m2_db.ordens

# mesma associação na tabela ponte
qtd = session.execute(select(func.count()).select_from(ordem_mecanico)).scalar_one()
assert qtd == 2
print("Linhas em ordem_mecanico:", qtd)
print("OK — relação N:M OrdemServico-Mecanico")

## 7. Consultas sobre a camada de persistência

In [ ]:
# Ordens por placa
ordens_placa = session.scalars(
    select(OrdemServico).join(Veiculo).where(Veiculo.placa == "ABC1D23")
).all()
print("Ordens da placa ABC1D23:", ordens_placa)

# Total de itens
n_servicos = session.scalar(select(func.count()).select_from(ItemServico))
n_pecas = session.scalar(select(func.count()).select_from(ItemPeca))
print(f"Itens serviço={n_servicos}, itens peça={n_pecas}")

# Valor recalculado
esperado = 120 + 90 + 45 + (80 * 2)
assert ordem_db.valor_total == esperado
print(f"Valor conferido: R$ {ordem_db.valor_total:.2f} (esperado {esperado})")
print("OK — consultas e cálculo de valor")

## 8. (Opcional) Executar a demo Java/JPA

A célula abaixo roda `mvn exec:java` na raiz do projeto (requer JDK 21+ e Maven).


In [ ]:
import subprocess
from pathlib import Path

root = Path("..").resolve() if Path.cwd().name == "notebooks" else Path.cwd()
result = subprocess.run(
    ["mvn", "-q", "exec:java"],
    cwd=root,
    capture_output=True,
    text=True,
)
print(result.stdout[-2000:] if result.stdout else "(sem stdout)")
if result.returncode != 0:
    print(result.stderr[-2000:])
print("exit:", result.returncode)

## Conclusão

| Relação | Entidades | Resultado |
|---------|-----------|-----------|
| 1:N | Cliente → Veiculo | Validada |
| 1:1 | OrdemServico ↔ NotaFiscal | Validada (FK única) |
| N:M | OrdemServico ↔ Mecanico | Validada (`ordem_mecanico`) |

Camada Java/JPA equivalente: pacote `br.ufg.oficina.model` + DAOs em `br.ufg.oficina.dao`.
